# Módulo 5 · Clase 2 — El pozo avisa antes de romperse

### Machine Learning for Petroleum Engineers Using Python
**SLB Ecuador · UDLA · 2026** — Carlos Enrique Mosquera Trujillo

Repositorio: https://github.com/cmosquerat/slb-diplomado

---

## La idea de hoy

Un pozo submarino tiene una válvula, el **choke**, que se va tapando sola con sal.
Cuando la alarma de la sala de control suena, el pozo ya está en falla.

**La pregunta: ¿cuánto antes se puede saber?**

1. Miramos los datos **con calma** antes de tocar un modelo (esa parte es la más larga).
2. Medimos la alarma que ya existe. Ese es el número a batir.
3. Convertimos la señal en una tabla.
4. Entrenamos, validamos con un pozo que el modelo nunca vio, y comparamos.

> **Antes de empezar:** este cuaderno se corre de arriba hacia abajo, una celda a la vez.
> Cada celda de código está comentada línea por línea. Si algo no se entiende, es un
> error nuestro: pregunten.

---

# 0 · Preparación

Cargamos las librerías. Cada una ya la usaron antes, y acá va de dónde viene.

In [ ]:
# pandas: tablas. Es el "Excel de Python".               (Modulo 1)
import pandas as pd

# numpy: cuentas con muchos numeros a la vez.            (Modulo 1)
import numpy as np

# matplotlib: graficos.                                  (Modulo 1)
import matplotlib.pyplot as plt

# el bosque de arboles que ya usamos para clasificar.    (Modulo 3 - Clase 3)
from sklearn.ensemble import RandomForestClassifier

# la particion honesta que deja un grupo entero afuera.  (Modulo 3 - Clase 5)
from sklearn.model_selection import GroupKFold

# para que los graficos salgan mas grandes y legibles
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Listo. Librerias cargadas.")

### Los datos

El archivo se baja solo desde GitHub. No hay que subir nada a Colab.

In [ ]:
# la direccion del archivo en el repositorio del curso
URL = ("https://raw.githubusercontent.com/cmosquerat/slb-diplomado/"
       "main/datos/pozos_3w_incrustacion.csv")

# read_csv lo baja y lo convierte en una tabla de pandas
d = pd.read_csv(URL)

# .shape devuelve (cuantas filas, cuantas columnas)
print("filas y columnas:", d.shape)

---

# 1 · Exploración de los datos  ⏱️ *15–20 minutos*

Esta es la parte más importante del cuaderno y la que más tiempo va a llevar.

**Por qué:** cada decisión que tomemos después — qué columnas usar, cómo partir los
datos, qué pozo dejar afuera — sale de acá. Si exploramos mal, el modelo va a estar
bien programado y mal planteado, que es la peor combinación posible.

Vamos a hacernos **ocho preguntas**, en orden, y cada una va a dejar una decisión escrita.

## 1.1 · ¿Cómo se ve una fila?

Lo primero, siempre: mirar el dato con los ojos.

In [ ]:
# .head(n) muestra las primeras n filas
d.head(5)

In [ ]:
# .tail(n) muestra las ULTIMAS n filas.
# Conviene mirar las dos puntas: a veces el final del archivo trae basura.
d.tail(5)

## 1.2 · ¿Qué es cada columna?

| columna | qué es | unidad |
|---|---|---|
| `pozo` | qué pozo es | — |
| `instancia` | qué grabación es (un pozo puede tener varias) | — |
| `t_min` | minutos desde que arrancó esa grabación | minutos |
| `p_antes_choke` | presión **justo antes** de la válvula | bar |
| `t_despues_choke` | temperatura **justo después** de la válvula | °C |
| `p_arbol` | presión en el árbol submarino (la boca del pozo) | bar |
| `p_anular` | presión entre las dos tuberías del pozo | bar |
| `p_gaslift` | presión del gas que se inyecta para aligerar | bar |
| `etiqueta` | **lo que hay que predecir**: `normal`, `transitorio` o `falla` | — |

Las dos primeras de presión/temperatura son las importantes:
**cuando el choke se tapa, la de arriba sube y la de abajo baja.**

In [ ]:
# .dtypes dice de que TIPO es cada columna.
# "object" = texto,  "float64" = numero con decimales,  "int64" = numero entero.
d.dtypes

> 🤔 **Pregunta clave**: `etiqueta` es texto y `p_antes_choke` es número.
> ¿Cuál de las dos puede entrar directo a un modelo, y cuál hay que traducir antes?

## 1.3 · ¿Cuánto dato tenemos, y de cuántos pozos?

Una cosa es tener 37 mil filas de **un** pozo, y otra muy distinta tenerlas de **cinco**.
Para el modelo, lo segundo vale muchísimo más.

In [ ]:
# .nunique() cuenta cuantos valores DISTINTOS hay en una columna
print("pozos distintos:      ", d.pozo.nunique())
print("grabaciones distintas:", d.instancia.nunique())

# cada fila es una medicion cada 30 segundos -> pasamos a horas
horas_totales = len(d) * 30 / 3600
print("horas de pozo grabadas:", round(horas_totales, 1))

In [ ]:
# ¿cuantas filas aporta cada pozo?
# .value_counts() cuenta cuantas veces aparece cada valor
d.pozo.value_counts()

In [ ]:
# Lo mismo pero en horas, que se entiende mejor que "filas".
# Recorremos pozo por pozo con un for, que es mas facil de leer que una linea magica.
for nombre in sorted(d.pozo.unique()):
    filas = len(d[d.pozo == nombre])            # las filas de ESE pozo
    horas = filas * 30 / 3600                   # a horas
    grabaciones = d[d.pozo == nombre].instancia.nunique()
    print(f"{nombre}: {horas:6.1f} horas en {grabaciones} grabacion(es)")

> 🔧 **Mini-ejercicio 1**: ¿cuántas horas tiene la grabación más larga del archivo?
> Pista: agrupen por `instancia` y cuenten filas.

In [ ]:
# Escribe tu solucion aqui

## 1.4 · ¿Qué dicen las etiquetas?

`etiqueta` es lo que queremos predecir. Antes de predecir nada hay que saber
**cuánto de cada cosa hay**. Si el 99 % fuera `normal`, el problema sería otro.

In [ ]:
# cuantas filas de cada etiqueta
conteo = d.etiqueta.value_counts()
print(conteo)
print()

# lo mismo en porcentaje, que es mas facil de interpretar
print((100 * conteo / len(d)).round(1))

In [ ]:
# Un grafico de barras se lee mas rapido que una tabla.
orden = ["normal", "transitorio", "falla"]
valores = [len(d[d.etiqueta == e]) for e in orden]      # filas de cada etiqueta
colores = ["#16A34A", "#D97706", "#C82B40"]             # verde, ambar, rojo

plt.figure(figsize=(7, 3.2))
plt.bar(orden, valores, color=colores)
plt.ylabel("filas (cada una = 30 segundos)")
plt.title("Cuanto tiempo pasa el pozo en cada estado")
plt.show()

**Lectura**: la mayor parte del archivo es `transitorio`, o sea el evento **avanzando**.
Eso no es casualidad: el dataset fue armado a propósito alrededor de eventos.

Hay muy poco `falla` porque la grabación normalmente se corta cuando el pozo ya falló
— para entonces ya no hay nada que estudiar.

> 🤔 **Pregunta clave**: si quisiéramos avisar **a tiempo**, ¿qué etiqueta nos interesa cazar?

## 1.5 · ¿Qué sensores existen de verdad?

Ésta es la pregunta que más veces se saltea, y la que más caro sale.
Que una columna esté en el archivo **no significa** que el sensor exista.

In [ ]:
# .isna() marca True donde falta el dato. .sum() los cuenta.
d.isna().sum()

In [ ]:
# La cuenta global no alcanza: hay que ver POR GRABACION.
# Un sensor puede existir en un pozo y no existir en otro.
sensores = ["p_antes_choke", "t_despues_choke", "p_arbol", "p_anular", "p_gaslift"]

print("¿que sensores tiene cada grabacion?\n")
for nombre in sorted(d.instancia.unique()):
    g = d[d.instancia == nombre]                 # solo esa grabacion
    vivos = []
    for s in sensores:
        # si la columna NO esta toda vacia, el sensor existe
        if not g[s].isna().all():
            vivos.append(s)
    print(f"{nombre}: {len(vivos)} sensores -> {', '.join(vivos)}")

**Lectura**: el **pozo 23** tiene solo **dos** sensores vivos. Los otros tres están en
el archivo pero completamente vacíos.

Esto no es un defecto del dataset: es cómo llegan los datos de campo. Sensores que
nunca se instalaron, que se rompieron, o que dejaron de reportar.

> 📌 **Decisión 1**: el pozo 23 sale de la clase y se convierte en el reto de la práctica.
> Trabajamos con los pozos 1, 21, 22 y 24, que tienen los cinco sensores.

In [ ]:
# nos quedamos con los cuatro pozos completos
pozos_clase = ["WELL-00001", "WELL-00021", "WELL-00022", "WELL-00024"]

# .isin() pregunta "¿esta en esta lista?"
dc = d[d.pozo.isin(pozos_clase)].copy()

print("filas que quedan:", len(dc))
print("grabaciones:     ", dc.instancia.nunique())

## 1.6 · ¿En qué rango se mueve cada sensor?

Antes de comparar nada, hay que saber con qué números estamos tratando.

In [ ]:
# .describe() da, por columna: cuantos datos hay, el promedio,
# la desviacion, el minimo, los cuartiles y el maximo.
# .T lo pone de costado para que se lea mejor.
dc[sensores].describe().T.round(2)

> 🤔 **Pregunta clave**: miren la columna `std` (desviación estándar).
> ¿Qué sensor es el que **menos** se mueve? ¿Eso lo hace inútil, o al contrario?

## 1.7 · El problema que va a decidir toda la clase

Miremos la presión antes del choke **pozo por pozo**.

In [ ]:
# El promedio de p_antes_choke en cada pozo.
# groupby("pozo") separa por pozo, y luego pedimos promedio, minimo y maximo.
dc.groupby("pozo")["p_antes_choke"].agg(["mean", "min", "max"]).round(1)

In [ ]:
# El mismo dato, dibujado. Una grabacion por linea.
plt.figure(figsize=(11, 4))
colores_pozo = {"WELL-00001": "#C82B40", "WELL-00021": "#2563EB",
                "WELL-00022": "#16A34A", "WELL-00024": "#EA580C"}

ya_dibujados = []                                  # para no repetir la leyenda
for nombre in sorted(dc.instancia.unique()):
    g = dc[dc.instancia == nombre]
    pozo = g.pozo.iloc[0]                          # a que pozo pertenece
    etiqueta = pozo if pozo not in ya_dibujados else None
    plt.plot(g.t_min, g.p_antes_choke, lw=1,
             color=colores_pozo[pozo], alpha=0.8, label=etiqueta)
    ya_dibujados.append(pozo)

plt.xlabel("minutos desde el inicio de la grabacion")
plt.ylabel("presion antes del choke [bar]")
plt.title("El mismo sensor, en cuatro pozos distintos")
plt.legend()
plt.show()

**Lectura — y esto es lo más importante del cuaderno**:

Un pozo trabaja alrededor de **54 bar** y otro alrededor de **130 bar**. No están mal
medidos: son pozos distintos, a profundidades distintas, con presiones de yacimiento
distintas.

Si le damos estos números crudos a un modelo, va a aprender a reconocer **pozos**,
no **fallas**. Y con un pozo nuevo no va a servir para nada.

> 📌 **Decisión 2**: hay que comparar cada pozo **consigo mismo** antes de compararlo
> con los demás. Cómo se hace, en la sección 3.

## 1.8 · Mirar una grabación entera, con calma

Los promedios esconden todo. Vamos a mirar **una** grabación completa, sensor por sensor.

In [ ]:
# elegimos una grabacion para mirarla en detalle
ejemplo = "WELL-00001_20170226130146"
g = dc[dc.instancia == ejemplo].reset_index(drop=True)

print("grabacion:", ejemplo)
print("dura", round(len(g) * 30 / 3600, 1), "horas")
print()
print(g.etiqueta.value_counts())

In [ ]:
# Una funcion chica para pintar el fondo segun la etiqueta.
# Asi vemos DONDE pasa cada cosa, sin tener que adivinar.
def pintar_fondo(g):
    colores = {"normal": "#DCFCE7", "transitorio": "#FEF3C7", "falla": "#FEE2E2"}
    etiquetas = g.etiqueta.values
    tiempos = g.t_min.values
    i = 0
    while i < len(etiquetas):
        j = i
        # avanzamos mientras la etiqueta no cambie
        while j + 1 < len(etiquetas) and etiquetas[j + 1] == etiquetas[i]:
            j = j + 1
        plt.axvspan(tiempos[i], tiempos[j], color=colores[etiquetas[i]], zorder=0)
        i = j + 1

# probamos la funcion con el sensor principal
plt.figure(figsize=(11, 3.5))
pintar_fondo(g)
plt.plot(g.t_min, g.p_antes_choke, color="#C82B40", lw=1.2)
plt.xlabel("minutos")
plt.ylabel("presion antes\ndel choke [bar]")
plt.title("Verde = normal    Amarillo = el evento avanzando    Rojo = falla")
plt.show()

In [ ]:
# Ahora los CINCO sensores, uno debajo del otro, en la misma grabacion.
nombres_bonitos = {
    "p_antes_choke":   "Presion antes del choke [bar]",
    "t_despues_choke": "Temperatura tras el choke [C]",
    "p_arbol":         "Presion en el arbol [bar]",
    "p_anular":        "Presion en el anular [bar]",
    "p_gaslift":       "Presion del gas lift [bar]",
}

fig, ejes = plt.subplots(5, 1, figsize=(11, 10), sharex=True)
for eje, s in zip(ejes, sensores):
    plt.sca(eje)                       # "dibuja en este panel"
    pintar_fondo(g)
    eje.plot(g.t_min, g[s], lw=1.1, color="#2D2D2D")
    eje.set_ylabel(nombres_bonitos[s], fontsize=9)

ejes[-1].set_xlabel("minutos desde el inicio de la grabacion")
fig.suptitle("Los cinco sensores del pozo WELL-00001", y=0.995)
plt.tight_layout()
plt.show()

**Lectura física**: fíjense en los dos primeros paneles y recuerden la cadena que vimos
en las láminas:

- la **presión antes del choke sube** → el fluido empuja contra un paso más chico;
- la **temperatura después del choke baja** → mayor caída de presión, mayor expansión,
  más frío (efecto Joule–Thomson).

Las dos cosas pasan **al mismo tiempo**, y esa combinación es la firma de una restricción
que **nadie ordenó**.

> 🔧 **Mini-ejercicio 2**: repitan el gráfico de los cinco sensores para la grabación
> `"WELL-00024_20160825100303"`. ¿Se ve la misma firma?

In [ ]:
# Escribe tu solucion aqui

## 1.9 · ¿Cuánto cambian los sensores entre normal y evento?

Ya lo vimos en una grabación. ¿Pasa en todas?

In [ ]:
# Comparamos el promedio de cada sensor en normal contra el resto.
# Primero creamos una columna simple: ¿es normal, si o no?
dc["es_evento"] = (dc.etiqueta != "normal")

# groupby nos da el promedio de cada sensor en cada grupo
resumen = dc.groupby("es_evento")[sensores].mean().round(2)
resumen.index = ["normal", "evento"]
resumen.T

**Lectura**: los promedios globales dicen poco, porque mezclan pozos con presiones muy
distintas. Este es exactamente el error que la Decisión 2 nos va a evitar.

Para verlo bien, hay que comparar **dentro de cada pozo**:

In [ ]:
# Ahora si: la diferencia normal -> evento, DENTRO de cada pozo.
print("cambio promedio del normal al evento, pozo por pozo\n")
for nombre in pozos_clase:
    g_pozo = dc[dc.pozo == nombre]
    normal = g_pozo[~g_pozo.es_evento]              # ~ significa "no"
    evento = g_pozo[g_pozo.es_evento]
    dp = evento.p_antes_choke.mean() - normal.p_antes_choke.mean()
    dt = evento.t_despues_choke.mean() - normal.t_despues_choke.mean()
    print(f"{nombre}:  presion {dp:+6.2f} bar     temperatura {dt:+6.2f} C")

**Lectura**: en los cuatro pozos la presión sube y la temperatura baja. Distinta
magnitud, misma dirección. **La física es la misma; los números no.**

## 1.10 · ¿Los sensores se mueven juntos?

Si dos sensores dicen exactamente lo mismo, uno de los dos sobra.

In [ ]:
# .corr() mide si dos columnas suben y bajan juntas.
#   +1 = se mueven igual,  -1 = se mueven al reves,  0 = no tienen relacion.
# Lo hacemos DENTRO de una grabacion, para no mezclar pozos.
g[sensores].corr().round(2)

In [ ]:
# La misma tabla, en colores: se lee de un vistazo.
matriz = g[sensores].corr()

plt.figure(figsize=(6, 5))
plt.imshow(matriz, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="correlacion")
plt.xticks(range(len(sensores)), sensores, rotation=45, ha="right", fontsize=8)
plt.yticks(range(len(sensores)), sensores, fontsize=8)

# escribimos el numero dentro de cada celda
for i in range(len(sensores)):
    for j in range(len(sensores)):
        plt.text(j, i, round(matriz.iloc[i, j], 2),
                 ha="center", va="center", fontsize=9)
plt.title("¿Que sensores se mueven juntos?")
plt.tight_layout()
plt.show()

**Lectura**: la presión antes del choke y la temperatura después están **fuertemente
correlacionadas, y con signo negativo**: cuando una sube, la otra baja. Es justo lo
que predice la física del choke.

> 🤔 **Pregunta clave**: si dos sensores tuvieran correlación +0.99, ¿los dejarían
> los dos en el modelo? ¿Qué se gana y qué se pierde?

## 1.11 · ¿Cuánto dura la ventana de aviso?

Ésta es la pregunta del negocio disfrazada de pregunta de datos: **cuánto tiempo
hay entre que el evento empieza y la grabación termina**. Ese es, como mucho, el
tiempo que podríamos ganar.

In [ ]:
print("ventana de oportunidad en cada grabacion\n")
for nombre in sorted(dc.instancia.unique()):
    gg = dc[dc.instancia == nombre]
    normal = gg[gg.etiqueta == "normal"]
    evento = gg[gg.etiqueta != "normal"]
    if len(evento) == 0:
        continue
    minuto_inicio = evento.t_min.min()          # cuando arranca el evento
    minuto_final = gg.t_min.max()               # cuando termina la grabacion
    print(f"{nombre}: normal hasta el minuto {minuto_inicio:6.0f}, "
          f"evento durante {minuto_final - minuto_inicio:6.0f} min")

> 📌 **Decisión 3**: nuestra métrica no va a ser "porcentaje de aciertos". Va a ser
> **cuántos minutos después del inicio del evento llega el primer aviso**. Menos es mejor.

## 1.12 · Cierre de la exploración

| Lo que vimos | Lo que decidimos |
|---|---|
| El pozo 23 tiene solo 2 sensores | Sale de la clase, es la práctica |
| Cada pozo trabaja en un rango de presión distinto | Comparar cada pozo **consigo mismo** |
| Presión sube y temperatura baja **a la vez** | Usar los dos, no uno solo |
| La mayoría del archivo es `transitorio` | Es lo que queremos cazar: ahí todavía se puede actuar |
| El evento dura horas | La métrica es **minutos de retraso**, no porcentaje |

Recién ahora podemos empezar a modelar. Todo lo que sigue son consecuencias de esta tabla.

---

# 2 · La alarma que ya existe

Antes de proponer algo nuevo hay que **medir lo que ya está instalado**. Si no,
no sabemos si mejoramos o empeoramos.

Una alarma de umbral funciona así:
1. mira una hora de operación normal y anota el valor típico;
2. anota también cuánto se mueve alrededor de ese valor;
3. si la señal se aleja más de **3 veces** ese movimiento típico, suena.

In [ ]:
# Trabajamos con la grabacion de ejemplo que ya conocemos.
# Las primeras 120 filas son la primera hora (120 x 30 s = 3600 s).
primera_hora = g.p_antes_choke.iloc[:120]

# el valor tipico. Usamos la MEDIANA y no el promedio porque
# un solo pico raro no la mueve.
centro = primera_hora.median()

# cuanto se mueve normalmente alrededor de ese centro
ancho = primera_hora.std()

print("valor tipico de este pozo:", round(centro, 2), "bar")
print("cuanto se mueve:          ", round(ancho, 3), "bar")

In [ ]:
# Ahora traducimos toda la señal a "cuantos anchos de distancia".
# A esto se le llama normalizar, y es LA idea de todo el cuaderno.
z = (g.p_antes_choke - centro) / ancho

# la alarma suena cuando se pasa de 3
suena = z.abs() > 3

print("la alarma suena en", suena.sum(), "de", len(z), "instantes")

In [ ]:
# Lo dibujamos: la señal normalizada, la banda, y donde suena.
plt.figure(figsize=(11, 4))
pintar_fondo(g)
plt.plot(g.t_min, z, color="#2D2D2D", lw=1.1)
plt.axhline(3, color="#2563EB", ls="--")      # borde de arriba
plt.axhline(-3, color="#2563EB", ls="--")     # borde de abajo
plt.axhspan(-3, 3, color="#E0E7FF", alpha=0.6, zorder=1)

# la primera vez que suena, DESPUES de que el evento empezo
inicio_evento = g.loc[g.etiqueta != "normal", "t_min"].min()
suena_tarde = g.t_min[(suena) & (g.t_min >= inicio_evento)]
if len(suena_tarde) > 0:
    plt.axvline(suena_tarde.iloc[0], color="#C82B40", lw=2)
plt.axvline(inicio_evento, color="#6B1525", lw=2, ls="--")

plt.ylabel("distancia a su normal\n(en 'anchos')")
plt.xlabel("minutos")
plt.title("Linea punteada oscura = empieza el evento    |    Linea roja = suena la alarma")
plt.show()

print("el evento empezo en el minuto:", round(inicio_evento))
print("la alarma sono en el minuto:  ", round(suena_tarde.iloc[0]))
print("llego", round(suena_tarde.iloc[0] - inicio_evento), "minutos tarde")

> 🔧 **Mini-ejercicio 3**: la industria usa 3 por costumbre, no por teorema.
> Cambien el 3 por un 2 y por un 4. ¿Qué pasa con el retraso? ¿Y con la cantidad
> de veces que suena durante el período normal?

In [ ]:
# Escribe tu solucion aqui

### Medir la alarma en las diez grabaciones

Una grabación no alcanza para juzgar nada. Vamos a las diez.

In [ ]:
def medir(tabla, aviso):
    """Devuelve los tres numeros que definimos en clase.

    tabla : la tabla con las columnas instancia, t_min, es_evento
    aviso : una lista de True/False, del mismo largo, que dice si avisa
    """
    t = tabla.copy()
    t["aviso"] = aviso

    detectadas = 0          # en cuantas grabaciones aviso alguna vez
    retrasos = []           # cuantos minutos tardo en cada una

    for nombre in sorted(t.instancia.unique()):
        gg = t[t.instancia == nombre]
        eventos = gg[gg.es_evento]
        if len(eventos) == 0:
            continue
        inicio = eventos.t_min.min()                       # arranca el evento
        avisos = gg[(gg.aviso) & (gg.t_min >= inicio)]     # avisos posteriores
        if len(avisos) > 0:
            detectadas = detectadas + 1
            retrasos.append(avisos.t_min.iloc[0] - inicio)

    # falsas alarmas: avisos durante el periodo normal
    normal = t[~t.es_evento]
    falsas = 100 * normal.aviso.sum() / len(normal)

    return {"detecta": detectadas,
            "de": t.instancia.nunique(),
            "retraso_min": float(np.median(retrasos)) if retrasos else float("nan"),
            "falsas_%": round(falsas, 1)}

print("funcion lista")

In [ ]:
# Aplicamos la alarma a TODAS las grabaciones, una por una.
partes = []
for nombre in sorted(dc.instancia.unique()):
    gg = dc[dc.instancia == nombre].sort_values("t_min").reset_index(drop=True)
    v = gg.p_antes_choke
    centro = v.iloc[:120].median()          # su propia normal
    ancho = v.iloc[:120].std()
    gg["z"] = (v - centro) / ancho
    partes.append(gg)

# pd.concat pega todas las partes en una sola tabla
alarma = pd.concat(partes, ignore_index=True)

# el resultado de la alarma
resultado_alarma = medir(alarma, alarma.z.abs() > 3)
print("LA ALARMA QUE YA EXISTE:")
print(resultado_alarma)

**Lectura**: éste es el récord a batir. La alarma **no es mala** — detecta la mayoría.
El problema es **cuándo** avisa.

Y hay un número que hay que anotar: el porcentaje de **falsas alarmas**. Ese va a ser
el presupuesto que ningún modelo puede pasar, porque si no la comparación es trampa.

In [ ]:
# guardamos el presupuesto de falsas alarmas para usarlo despues
PRESUPUESTO = resultado_alarma["falsas_%"]
print("presupuesto de falsas alarmas:", PRESUPUESTO, "%")

---

# 3 · De la señal a una tabla

Un modelo no come señales: come **tablas**. Filas independientes, donde cada fila
se explica sola.

El problema es que en una señal el significado de un número **depende de los que
vinieron antes**. 54 bar no dice nada; 54 bar *después de venir de 53 durante ocho
horas* lo dice todo.

Entonces hay que meterle a cada fila, en sus columnas, el contexto que tenía en el tiempo.
Vamos a construir **cuatro columnas por sensor**:

| columna | qué es | cómo se calcula |
|---|---|---|
| `__base` | la señal comparada con su propia normal | `(v - centro) / ancho` |
| `__nivel` | dónde está parada, sin el temblor | promedio de la última media hora |
| `__ruido` | cuánto tiembla | desviación de esa media hora |
| `__pend` | hacia dónde va | valor de ahora menos el de media hora atrás |

In [ ]:
# 60 filas = 60 x 30 s = 30 minutos. Esa es nuestra ventana.
VENTANA = 60

# 120 filas = 1 hora. Esa es la "normal" de cada pozo.
BASE = 120


def preparar(g):
    """Convierte UNA grabacion en filas de tabla."""
    X = pd.DataFrame(index=g.index)

    for s in sensores:
        v = g[s]

        # --- paso 1: cual es la normal de ESTE pozo ---
        centro = v.iloc[:BASE].median()      # su valor tipico
        ancho = v.iloc[:BASE].std()          # cuanto se mueve normalmente

        # --- paso 2: la señal en "distancias a su propia normal" ---
        z = (v - centro) / ancho
        X[s + "__base"] = z

        # --- paso 3: las tres componentes, calculadas sobre z ---
        # .rolling(n) toma ventanas de n filas que se van corriendo
        X[s + "__nivel"] = z.rolling(VENTANA).mean()   # el promedio
        X[s + "__ruido"] = z.rolling(VENTANA).std()    # el temblor
        X[s + "__pend"] = z.diff(VENTANA)              # el cambio

    return X

print("funcion lista")

In [ ]:
# La aplicamos a todas las grabaciones y armamos LA tabla del modelo.
partes = []
for nombre in sorted(dc.instancia.unique()):
    gg = dc[dc.instancia == nombre].sort_values("t_min").reset_index(drop=True)

    X = preparar(gg)                       # las columnas nuevas

    # guardamos tambien el valor CRUDO de cada sensor: lo vamos a
    # necesitar en la seccion 5 para comparar contra el
    for s in sensores:
        X[s + "__crudo"] = gg[s]

    X["y"] = gg.es_evento.astype(int)      # lo que hay que predecir: 0 o 1
    X["t_min"] = gg.t_min
    X["instancia"] = nombre
    X["pozo"] = gg.pozo.iloc[0]
    X["es_evento"] = gg.es_evento

    # las primeras filas no tienen ventana completa: se descartan
    partes.append(X.dropna())

T = pd.concat(partes, ignore_index=True)

print("la tabla del modelo:", T.shape)
print()
T.head(3)

In [ ]:
# ¿Como quedaron las columnas? Cuatro por cada uno de los cinco sensores.
columnas_modelo = [c for c in T.columns
                   if c.endswith(("__base", "__nivel", "__ruido", "__pend"))]
print("columnas creadas:", len(columnas_modelo))
for c in columnas_modelo[:8]:
    print("   ", c)
print("    ...")

In [ ]:
# Veamos que hizo cada componente, sobre la grabacion de ejemplo.
te = T[T.instancia == ejemplo]

fig, ejes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
piezas = [("p_antes_choke__base",  "La señal contra su propia normal", "#2D2D2D"),
          ("p_antes_choke__nivel", "El NIVEL: sube y no vuelve",       "#C82B40"),
          ("p_antes_choke__ruido", "El RUIDO: ademas empieza a temblar", "#2563EB")]

for eje, (col, titulo, color) in zip(ejes, piezas):
    eje.plot(te.t_min, te[col], color=color, lw=1.3)
    eje.set_title(titulo, loc="left", fontsize=10)

ejes[-1].set_xlabel("minutos")
plt.tight_layout()
plt.show()

> 🔧 **Mini-ejercicio 4**: cambien `VENTANA` de 60 a 20 (10 minutos) y vuelvan a
> dibujar. ¿El nivel queda más suave o más tembloroso? ¿Qué se gana y qué se pierde
> al agrandar la ventana?

In [ ]:
# Escribe tu solucion aqui

---

### Una corrección antes de comparar

La tabla `T` perdió las primeras filas de cada grabación: las que todavía no tenían
media hora de ventana detrás. La alarma, en cambio, la medimos sobre **todas** las filas.

Comparar dos cosas medidas sobre poblaciones distintas es un error clásico y silencioso.
Así que volvemos a medir la alarma **sobre las mismas filas** que va a ver el modelo.

In [ ]:
# la misma alarma de 3 desviaciones, pero sobre la tabla T
aviso_alarma = T["p_antes_choke__base"].abs() > 3

resultado_alarma = medir(T, aviso_alarma)
PRESUPUESTO = resultado_alarma["falsas_%"]

print("LA ALARMA, medida sobre las mismas filas que vera el modelo:")
print(resultado_alarma)
print()
print("presupuesto de falsas alarmas:", PRESUPUESTO, "%")

---

# 4 · El modelo, y cómo validarlo sin engañarnos

Vamos a usar **Random Forest**, el mismo del Módulo 3 · Clase 3.

**Recordatorio, porque nada se da por sabido**: un árbol de decisión es una cadena de
preguntas de sí/no aprendidas del dato (*¿la temperatura bajó más de 1.2? ¿y la presión
subió más de 0.8?*). Un árbol solo es frágil. Un **bosque** entrena 300 árboles, cada
uno viendo una porción distinta de los datos, y los hace **votar**.

### Cómo se valida, y cómo NO

❌ **Así no**: mezclar todas las filas y apartar el 20 % al azar. Cada pozo quedaría a
los dos lados. El modelo ya vio ese pozo, ya sabe cómo respira, y en el examen lo
reconoce. El número sale hermoso **y es mentira**.

✅ **Así sí**: sacar **un pozo entero**, entrenar con los otros tres, y preguntarle por
el que nunca vio. Repetir dejando cada pozo afuera por turno. Eso es `GroupKFold`
agrupando **por pozo**, y responde la pregunta real:
*«mañana lo instalamos en un pozo nuevo, ¿va a funcionar?»*

In [ ]:
def entrenar_y_evaluar(T, columnas):
    """Entrena dejando un POZO entero afuera cada vez.

    Devuelve, para cada fila, la probabilidad de evento que le asigno
    un modelo que NUNCA vio ese pozo.
    """
    # un lugar para guardar la respuesta de cada fila
    probabilidad = np.zeros(len(T))

    # GroupKFold parte los datos respetando los grupos (los pozos)
    particion = GroupKFold(n_splits=T.pozo.nunique())

    for filas_entrenar, filas_probar in particion.split(T[columnas], T.y, T.pozo):
        modelo = RandomForestClassifier(
            n_estimators=300,          # 300 arboles
            min_samples_leaf=5,        # no partir hojas con menos de 5 casos
            class_weight="balanced",   # compensar que hay mas evento que normal
            random_state=0,            # para que el resultado se repita
            n_jobs=-1,                 # usar todos los procesadores
        )
        modelo.fit(T[columnas].iloc[filas_entrenar], T.y.iloc[filas_entrenar])

        # predict_proba da la probabilidad. Nos quedamos con la del "1"
        probabilidad[filas_probar] = modelo.predict_proba(
            T[columnas].iloc[filas_probar])[:, 1]

    return probabilidad

print("funcion lista")

In [ ]:
# Entrenamos con las columnas "__base": la señal comparada con su propia normal.
columnas_base = [c for c in T.columns if c.endswith("__base")]
print("usando", len(columnas_base), "columnas:", columnas_base)

# esto tarda un poco: son 4 modelos de 300 arboles
prob = entrenar_y_evaluar(T, columnas_base)

print("listo. Ejemplo de probabilidades:", prob[:5].round(2))

### El modelo no dice sí o no: dice *qué tan seguro está*

Nosotros elegimos a partir de qué número suena. Y para comparar de forma justa contra
la alarma, tenemos que elegir el umbral que da **las mismas falsas alarmas**.

In [ ]:
def umbral_justo(T, prob, presupuesto):
    """Busca el umbral mas permisivo que no pasa el presupuesto de falsas alarmas."""
    for u in np.arange(0.20, 0.99, 0.005):
        r = medir(T, prob >= u)
        if r["falsas_%"] <= presupuesto:
            return round(float(u), 3)
    return 0.99

UMBRAL = umbral_justo(T, prob, PRESUPUESTO)
print("umbral elegido:", UMBRAL)
print()
print("EL MODELO:            ", medir(T, prob >= UMBRAL))
print("LA ALARMA QUE EXISTE: ", resultado_alarma)

**Lectura**: mismas falsas alarmas, más eventos detectados y aviso mucho más temprano.

Cada minuto de esa diferencia es un minuto en que alguien todavía puede abrir el choke.

---

# 5 · ¿Qué parte del mérito es del modelo?

Ésta es la pregunta que casi nunca se hace, y es la más importante de la clase.

Vamos a entrenar el **mismo** Random Forest cuatro veces, cambiando **solo** las
columnas que le damos, y comparar todo con el mismo presupuesto de falsas alarmas.

In [ ]:
# los cuatro escalones: que columnas le damos al modelo en cada uno
escalones = [
    ("1. Los numeros crudos del sensor", [s + "__crudo" for s in sensores]),
    ("2. Comparados con su propia normal", [c for c in T.columns if c.endswith("__base")]),
    ("3. Ademas, promediados (nivel)", [c for c in T.columns
                                        if c.endswith(("__base", "__nivel"))]),
    ("4. Ademas, ruido y pendiente",
     [c for c in T.columns if c.endswith(("__base", "__nivel", "__ruido", "__pend"))]),
]

resultados = []
for nombre, cols in escalones:
    p = entrenar_y_evaluar(T, cols)
    u = umbral_justo(T, p, PRESUPUESTO)
    r = medir(T, p >= u)
    r["escalon"] = nombre
    resultados.append(r)
    print(f"{nombre:38s} -> detecta {r['detecta']}/{r['de']}, "
          f"avisa a los {r['retraso_min']:.0f} min")

In [ ]:
# Todo junto, incluida la alarma, en una sola tabla
tabla_final = pd.DataFrame(
    [{"escalon": "0. La alarma que ya existe", **resultado_alarma}] + resultados
)
tabla_final = tabla_final[["escalon", "detecta", "de", "retraso_min", "falsas_%"]]
tabla_final

In [ ]:
# Y en un grafico, que se lee de un vistazo
plt.figure(figsize=(10, 4))
nombres = tabla_final.escalon.str[:26]
valores = tabla_final.detecta
colores = ["#9CA3AF", "#D1D5DB", "#C82B40", "#F0A0AC", "#E7CDD1"]

barras = plt.bar(range(len(valores)), valores, color=colores[:len(valores)])
for i, (b, r) in enumerate(zip(barras, tabla_final.itertuples())):
    plt.text(b.get_x() + b.get_width() / 2, r.detecta + 0.15,
             f"{r.detecta}/{r.de}", ha="center", fontweight="bold")
    plt.text(b.get_x() + b.get_width() / 2, r.detecta / 2,
             f"{r.retraso_min:.0f} min", ha="center", fontsize=9)

plt.xticks(range(len(valores)), nombres, rotation=20, ha="right", fontsize=8)
plt.ylabel("grabaciones detectadas")
plt.title("Todos con el mismo presupuesto de falsas alarmas")
plt.tight_layout()
plt.show()

**Lectura — el número del día**:

Entre el escalón 1 y el escalón 2 hay **dos líneas de código**: restar la mediana y
dividir por la desviación. El algoritmo es el mismo, los datos son los mismos, el día
es el mismo.

Y el resultado se da vuelta por completo.

Los escalones 3 y 4 agregan columnas y **no mejoran**. Eso también hay que decirlo:
si una idea no gana, se descarta, aunque suene sofisticada.

> **Lo que se llevan**: elegir bien *qué se le pregunta al dato* valió más que
> cualquier modelo. Y eso no lo hace la librería — lo hace alguien que entiende el pozo.

---

# 6 · ¿En qué se apoyó el modelo?

Random Forest puede decirnos qué columnas usó más. Vale la pena preguntarle,
porque a veces contesta algo de física.

In [ ]:
# entrenamos un modelo con TODOS los datos, solo para preguntarle
modelo = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                class_weight="balanced", random_state=0, n_jobs=-1)
modelo.fit(T[columnas_base], T.y)

# .feature_importances_ dice cuanto peso tuvo cada columna
importancia = pd.Series(modelo.feature_importances_, index=columnas_base)
importancia = importancia.sort_values()

plt.figure(figsize=(8, 3.5))
plt.barh(range(len(importancia)), 100 * importancia.values, color="#9CA3AF")
plt.yticks(range(len(importancia)),
           [c.replace("__base", "") for c in importancia.index], fontsize=9)
plt.xlabel("cuanto se apoya el modelo en cada sensor [%]")
plt.title("Quien delata la incrustacion")
plt.tight_layout()
plt.show()

**Lectura física**: el sensor que más pesa **no es la presión**: es la **temperatura**.

Tiene sentido. La presión antes del choke también sube cuando alguien cierra un poco
la válvula **a propósito**. La temperatura cayendo *mientras* la presión sube es la
firma de una restricción que **nadie ordenó**.

El modelo redescubrió solo el efecto Joule–Thomson. Y de paso nos dijo qué sensor no
conviene dejar sin mantenimiento.

## La perilla: no hay «un modelo», hay una decisión

In [ ]:
# Recorremos umbrales y vemos como se mueven las tres metricas
filas = []
for u in np.arange(0.25, 0.95, 0.05):
    r = medir(T, prob >= u)
    r["umbral"] = round(u, 2)
    filas.append(r)

curva = pd.DataFrame(filas)[["umbral", "detecta", "retraso_min", "falsas_%"]]
curva

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(curva["falsas_%"], curva.detecta, "o-", color="#C82B40")
for _, r in curva.iterrows():
    plt.annotate(f"{r.umbral:.2f}", (r["falsas_%"], r.detecta),
                 textcoords="offset points", xytext=(6, 5), fontsize=8)
plt.xlabel("falsas alarmas durante operacion normal [%]")
plt.ylabel("grabaciones detectadas")
plt.title("Girar la perilla es una decision de negocio, no tecnica")
plt.tight_layout()
plt.show()

> 🤔 **Pregunta clave**: ¿quién debería elegir ese umbral: el que programa el modelo,
> o el que va a atender la alerta a las tres de la mañana?

---

# 7 · Las falsas alarmas que no lo eran

El modelo avisa algunas veces durante el período marcado como `normal`. La métrica
las cuenta como error.

Pero: **¿están repartidas al azar, o se amontonan en algún lado?**

In [ ]:
# Partimos el periodo normal de cada grabacion en tres tercios
# y contamos que porcentaje del tiempo avisa en cada uno.
T["aviso"] = prob >= UMBRAL

tercios = [[], [], []]
for nombre in sorted(T.instancia.unique()):
    gg = T[(T.instancia == nombre) & (~T.es_evento)].sort_values("t_min")
    if len(gg) < 30:
        continue
    k = len(gg) // 3
    partes_g = [gg.iloc[:k], gg.iloc[k:2 * k], gg.iloc[2 * k:]]
    for i in range(3):
        tercios[i].append(100 * partes_g[i].aviso.mean())

promedios = [float(np.mean(x)) for x in tercios]

plt.figure(figsize=(7, 3.5))
plt.bar(["primer tercio", "segundo tercio", "ultimo tercio"], promedios,
        color=["#E5E7EB", "#FDE68A", "#C82B40"])
for i, v in enumerate(promedios):
    plt.text(i, v + 0.4, f"{v:.0f} %", ha="center", fontweight="bold")
plt.ylabel("avisos durante el periodo\netiquetado como normal [%]")
plt.title("¿Donde caen las 'falsas alarmas'?")
plt.tight_layout()
plt.show()

**Lectura**: si el modelo se equivocara al azar, las tres barras serían iguales.
**No lo son**: los avisos se amontonan justo **antes** de que el especialista marque
el inicio del evento.

Recuerden lo que dijimos en la lámina de las etiquetas: la incrustación no empieza
en un segundo, empieza de a poco, y una persona tuvo que **elegir** un segundo para
marcarla. Cuando el modelo avisa antes de esa marca, la métrica lo llama error.

> **La lección incómoda**: una métrica compara al modelo contra la **etiqueta**, no
> contra la realidad. Por eso ningún resultado se cierra sin ir a mirar **dónde** se
> equivoca.

---

# 🧩 Práctica: el pozo que dejamos afuera  ⏱️ *18 minutos*

El **pozo 23** tiene 141 horas grabadas y solo **dos** sensores vivos:
`p_antes_choke` y `p_gaslift`. Es el pozo más real de todos: así llegan los datos
cuando nadie los preparó para una clase.

**El primer paso ya está resuelto.** Los otros son de ustedes.

In [ ]:
# ── PASO 1 (RESUELTO) ── armamos la tabla del pozo 23 con los dos sensores que tiene
sensores_23 = ["p_antes_choke", "p_gaslift"]

partes = []
for nombre in sorted(d[d.pozo == "WELL-00023"].instancia.unique()):
    gg = d[d.instancia == nombre].sort_values("t_min").reset_index(drop=True)

    X = pd.DataFrame(index=gg.index)
    for s in sensores_23:
        v = gg[s]
        centro = v.iloc[:BASE].median()
        ancho = v.iloc[:BASE].std()
        X[s + "__base"] = (v - centro) / ancho

    X["y"] = (gg.etiqueta != "normal").astype(int)
    X["es_evento"] = gg.etiqueta != "normal"
    X["t_min"] = gg.t_min
    X["instancia"] = nombre
    X["pozo"] = "WELL-00023"
    partes.append(X.dropna())

T23 = pd.concat(partes, ignore_index=True)
print("tabla del pozo 23:", T23.shape)
print("grabaciones:", T23.instancia.nunique())
T23.head(3)

### Paso 2 — entrenen con los cuatro pozos de la clase y pregúntenle por el 23

Pista: entrenen un `RandomForestClassifier` con `T` usando **solo** las dos columnas
que el pozo 23 también tiene (`p_antes_choke__base` y `p_gaslift__base`), y después
usen `.predict_proba(...)` sobre `T23`.

In [ ]:
# Escribe tu solucion aqui

### Paso 3 — midan

Usen la función `medir(T23, aviso)` que ya está escrita.
¿Cuántas de las **dos** grabaciones detecta? ¿A los cuántos minutos?

In [ ]:
# Escribe tu solucion aqui

### Paso 4 — giren la perilla

Busquen el umbral que detecta las **dos** grabaciones.
¿Cuántas falsas alarmas costó llegar ahí?

In [ ]:
# Escribe tu solucion aqui

### Paso 5 — comparen contra la alarma de umbral

Corran la alarma de 3 desviaciones sobre `p_antes_choke` del pozo 23.
¿Le gana el modelo, aun con solo dos sensores?

In [ ]:
# Escribe tu solucion aqui

### Paso 6 — la pregunta de negocio

Escriban **una sola frase** para el jefe de operaciones que responda:

> *¿Instalamos esto en el pozo 23, o primero mandamos a arreglar los tres sensores
> muertos?*

La frase tiene que decir **qué se gana con cada opción**, en minutos de aviso.
No en porcentajes.

*(escriban su frase acá, en esta celda de texto)*

**Nuestra respuesta:**

---

# Cierre

| Lo que aprendimos | Dónde se usa mañana |
|---|---|
| Toda señal es **nivel + lo que se repite + ruido** | Cualquier sensor, cualquier equipo |
| Una **ventana deslizante** convierte una señal en una tabla | Todo dato de proceso |
| Comparar cada activo **consigo mismo** antes que entre activos | Flotas de pozos, bombas, compresores |
| Validar dejando **una unidad entera afuera** | Siempre que haya varios equipos |
| Una **etiqueta** es un juicio humano, no la verdad | Todo problema supervisado |

**El número del día: 74 minutos.** Lo que se ganó sin comprar un solo sensor nuevo.

---

### La próxima clase

**Módulo 5 · Clase 3 — Curvas de declinación:** Arps (1945), la deuda que quedó
anotada en la Clase 1. Exponencial, hiperbólica y el EUR. Ahí vuelve el ciclo
estacional, porque el dato va a ser mensual.

---

### Datos

**3W Dataset v2.0.0** — Petrobras · `github.com/petrobras/3W` · datos bajo CC BY 4.0.
Vargas, R. E. V. *et al.* (2019), *A realistic and public dataset with rare undesirable
real events in oil wells*, JPSE 181, 106223.